# Step 1 â€“ Loading the NDW Dexter data

In this notebook we read the 14 Excel files we exported from NDW Dexter and put all
measurements in **one table**, with one row per detector, date and hour.

We exported one file per Tuesday (25 Aug â€“ 6 Oct 2026) and per location group:
- **group A**: 20 detectors (most measurement points)
- **group B**: 12 detectors (the points around the N345, km 90.5/90.6)

The result is saved as `processed_data/A1_detectors_hourly.csv`, which the next notebooks use.

In [4]:
import pandas as pd
import glob
import os



## 1.1 What does a Dexter file look like?

Before writing code that reads all files, we first open one file to see how it is built.

In [5]:
example_file = "../data_dexter/A1_groepA_2026-09-15.xlsx"

# Which sheets are in the file?
print(pd.ExcelFile(example_file).sheet_names)

['Overzicht', 'Intensiteit', 'Gemiddelde snelheid', 'Datakwaliteit informatie']


In [6]:
# Read the sheet 'Intensiteit' without a header, so we see the raw rows
raw = pd.read_excel(example_file, sheet_name="Intensiteit", header=None)
raw.head(32)

,0,1,2,3,4,5
0,A1_groepA_2026-09-15,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,Gemiddelde voertuigverdeling per uur van 2026-...,NaN,NaN,NaN,NaN,NaN
4,uur op de dag,Intensiteit,onbepaald (%),NaN,NaN,NaN
5,00:00 - 00:59,234,-1,NaN,NaN,NaN
6,01:00 - 01:59,123.1,-1,NaN,NaN,NaN
7,02:00 - 02:59,121,-1,NaN,NaN,NaN
8,03:00 - 03:59,168,-1,NaN,NaN,NaN
9,04:00 - 04:59,240,-1,NaN,NaN,NaN


The sheet is not a normal table. It contains one **block per detector**:

1. a title row, for example
   `Gemiddelde voertuigverdeling per uur van 2026-09-15 00:00:00 tot 2026-09-15 23:59:59 voor 0011hrl0434ra (RWS01_MONIBAS_0011hrl0434ra) op werkdagen`
2. a header row (`uur op de dag`, `Intensiteit`, ...)
3. 24 rows with the hour (`07:00 - 07:59`) and the number of vehicles in that hour
4. some summary rows (`Totaal`, `7:00-8:59 uur`) that we do not need

The sheet `Gemiddelde snelheid` (average speed) has the same structure.
So we need to go through the rows one by one: when we see a title row we remember
the detector ID and the date, and when we see an hour row we save the value.

## 1.2 A function that reads one sheet

In [7]:
def read_dexter_sheet(path, sheet_name, value_name):
    """Read one sheet of a Dexter export into a table: detector, date, hour, value."""
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    rows = []
    detector = None
    date = None

    for text, value in zip(raw[0], raw[1]):
        if not isinstance(text, str):
            continue  # empty row

        # Title row of a new block: take the detector ID (between brackets) and the date
        if text.startswith("Gemiddelde voertuigverdeling"):
            detector = text.split("(")[-1].split(")")[0]
            date = text.split("van ")[1][:10]

        # Hour row, for example "07:00 - 07:59" (13 characters)
        elif detector is not None and len(text) == 13 and text[2:5] == ":00":
            hour = int(text[:2])
            rows.append([detector, date, hour, value])

    return pd.DataFrame(rows, columns=["detector", "date", "hour", value_name])

In [14]:
# Test the function on the example file
test = read_dexter_sheet(example_file, "Intensiteit", "intensity")
print(test.shape)                      # expected: 20 detectors x 24 hours = 480 rows
print(test["detector"].nunique())      # expected: 20
test.head(25)

(480, 4)
20


,detector,date,hour,intensity
0,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,0,234.0
1,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,1,123.1
2,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,2,121.0
3,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,3,168.0
4,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,4,240.0
5,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,5,1246.0
6,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,6,3523.0
7,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,7,3749.0
8,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,8,1826.0
9,RWS01_MONIBAS_0011hrl0434ra,2026-09-15,9,3242.0


## 1.3 Read all 14 files

For every file we read both sheets (intensity and speed) and merge them on detector, date and hour.

In [15]:
files = sorted(glob.glob("../data_dexter/*.xlsx"))
print(len(files), "files found")

tables = []
for f in files:
    intensity = read_dexter_sheet(f, "Intensiteit", "intensity")
    speed = read_dexter_sheet(f, "Gemiddelde snelheid", "speed")
    tables.append(intensity.merge(speed, on=["detector", "date", "hour"]))

data = pd.concat(tables, ignore_index=True)
print(data.shape)
data.head()

14 files found
(5360, 5)


,detector,date,hour,intensity,speed
0,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,0,265.3,115
1,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,1,180.0,107
2,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,2,148.6,106
3,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,3,142.9,104
4,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,4,290.0,104


## 1.4 Check the data

We expect 32 detectors Ã— 7 days Ã— 24 hours = 5376 rows. Let's check whether everything is there.

In [10]:
print("Number of detectors:", data["detector"].nunique())
print(data["date"].value_counts().sort_index())

Number of detectors: 32
date
2026-08-25    768
2026-09-01    768
2026-09-08    764
2026-09-15    768
2026-09-22    768
2026-09-29    764
2026-10-06    760
Name: count, dtype: int64


In [11]:
# Which detector-days do not have all 24 hours?
counts = data.groupby(["detector", "date"]).size()
counts[counts != 24]

detector                           date      
RWS01_MONICA_00D00106CC55D0070007  2026-09-08    23
                                   2026-10-06    21
RWS01_MONICA_00D00106CC55D007000B  2026-09-08    23
                                   2026-10-06    21
RWS01_MONICA_00D00106CC55D0070087  2026-09-08    23
                                   2026-10-06    22
RWS01_MONICA_00D00106CC55D007008B  2026-09-08    23
                                   2026-09-29    20
dtype: int64

A few hours are missing for the detectors at **Re 43.6** (8 Sep, 29 Sep and 6 Oct).
We keep this in mind for the next step: if we add up lanes, a missing lane makes the total too low.

In [12]:
# Dexter gives speed = 0 when no vehicles passed. Is that always the case?
print("Rows with speed 0:", (data["speed"] == 0).sum())
print("Highest intensity in those rows:", data.loc[data["speed"] == 0, "intensity"].max())

Rows with speed 0: 152
Highest intensity in those rows: 0.0


All rows with speed 0 also have intensity 0, so speed 0 means "no data", not standing traffic.
We will treat these as missing values when we calculate average speeds.

## 1.5 Save the result

In [13]:
os.makedirs("../processed_data", exist_ok=True)
data.to_csv("../processed_data/A1_detectors_hourly.csv", index=False)
print("Saved", len(data), "rows")

Saved 5360 rows
